# cuDF Checkpoint Benchmark: CPU vs GPU

This notebook benchmarks FlowBook's checkpoint overhead for cudf DataFrames,
comparing CPU-side (default) vs GPU-side checkpointing.

The benchmark uses a DataFrame large enough to show measurable differences:
- 4M rows, ~300 columns (mimicking a real Kaggle feature engineering pipeline)
- 229 int8 columns + 22 float32 + 51 float64
- On CPU, int8 columns get inflated to float64 during column-by-column conversion

In [ ]:
import cudf
import numpy as np
import time
import gc

## Build a Notebook-Scale DataFrame

This mirrors the column structure of a top Kaggle solution:
- 21 factorized categoricals (float32)
- 21×2 interaction features (float64)
- 210 pair features C(21,2) (int8)
- 9 digit + 10 digit-combo features (int8)

In [ ]:
n_rows = 4_000_000
rng = np.random.default_rng(42)
data = {}

# 21 factorized categoricals (float32)
for i in range(21):
    data[f'cat_{i}'] = rng.integers(0, 50, size=n_rows).astype(np.float32)

# Weight + Price (float64)
data['weight'] = rng.normal(50, 10, size=n_rows)
data['price'] = rng.normal(100, 20, size=n_rows)

# 21 nan_wc + 21 cat_wc (float64)
for i in range(21):
    data[f'cat_{i}_nan_wc'] = rng.normal(0, 1, size=n_rows)
    data[f'cat_{i}_wc'] = rng.normal(0, 1, size=n_rows)

# NaNs feature (float32)
data['NaNs'] = rng.integers(0, 100, size=n_rows).astype(np.float32)

# 7 float64 features
for i in range(7):
    data[f'extra_{i}'] = rng.normal(0, 1, size=n_rows)

# 9 digit features (int8)
for i in range(1, 10):
    data[f'digit{i}'] = rng.integers(-1, 10, size=n_rows).astype(np.int8)

# 10 digit-combo features (int8)
for i in range(4):
    for j in range(i + 1, 5):
        data[f'digit_{i+1}_{j+1}'] = rng.integers(0, 121, size=n_rows).astype(np.int8)

# 210 pair features (int8) — C(21,2)
cats = [f'cat_{i}' for i in range(21)]
for i in range(len(cats)):
    for j in range(i + 1, len(cats)):
        data[f'{cats[i]}_{cats[j]}'] = rng.integers(0, 127, size=n_rows).astype(np.int8)

df = cudf.DataFrame(data)

int8_cols = [c for c in df.columns if df[c].dtype == np.int8]
f32_cols = [c for c in df.columns if df[c].dtype == np.float32]
f64_cols = [c for c in df.columns if df[c].dtype == np.float64]

print(f"Shape: {df.shape}")
print(f"Columns: {len(int8_cols)} int8, {len(f32_cols)} float32, {len(f64_cols)} float64")
print(f"GPU memory: {df.memory_usage(deep=True).sum() / 1024**2:.0f} MB")

## Benchmark: Manual Checkpoint Timing

Before enabling GPU checkpoint mode, let's manually time the two approaches
to understand the performance difference.

In [ ]:
# CPU-side checkpoint: convert to pandas via batch path
# (This is the _fsproxy_slow path that preserves compact dtypes)
gc.collect()
t0 = time.perf_counter()
pandas_copy = df.to_pandas()
t_cpu = time.perf_counter() - t0
cpu_mb = pandas_copy.memory_usage(deep=True).sum() / 1024**2
print(f"CPU checkpoint (to_pandas): {t_cpu*1000:.0f} ms, {cpu_mb:.0f} MB")
del pandas_copy
gc.collect()

In [ ]:
# GPU-side checkpoint: deep copy on GPU
gc.collect()
t0 = time.perf_counter()
gpu_copy = df.copy(deep=True)
t_gpu = time.perf_counter() - t0
gpu_mb = gpu_copy.memory_usage(deep=True).sum() / 1024**2
print(f"GPU checkpoint (deep copy): {t_gpu*1000:.1f} ms, {gpu_mb:.0f} MB")
print(f"\nSpeedup: {t_cpu/t_gpu:.0f}x faster")
print(f"Memory: CPU={cpu_mb:.0f} MB, GPU={gpu_mb:.0f} MB")
del gpu_copy
gc.collect()

## Benchmark: GPU-Native Diffing

FlowBook detects changes between checkpoints. On GPU, this uses
`cudf.DataFrame.equals()` for the fast path (no changes) and
per-column comparison for the slow path (some columns changed).

In [ ]:
# GPU diff: no changes (fast path)
df_copy = df.copy(deep=True)

t0 = time.perf_counter()
equal = df.equals(df_copy)
t_equal = time.perf_counter() - t0
print(f"GPU equals (no changes): {t_equal*1000:.1f} ms — {equal}")

# GPU diff: one column changed
df_copy['price'] = df_copy['price'] * 1.1
t0 = time.perf_counter()
equal = df.equals(df_copy)
t_changed = time.perf_counter() - t0
print(f"GPU equals (1 col changed): {t_changed*1000:.1f} ms — {equal}")

# GPU diff: find which columns changed
t0 = time.perf_counter()
changed_cols = [c for c in df.columns if not df[c].equals(df_copy[c])]
t_find = time.perf_counter() - t0
print(f"GPU per-column scan ({len(df.columns)} cols): {t_find*1000:.0f} ms")
print(f"Changed columns: {changed_cols}")

del df_copy
gc.collect()

## Run with FlowBook GPU Checkpoint Mode

Now enable GPU checkpoint mode and let FlowBook handle the
checkpointing automatically during cell execution.

In [ ]:
%cudf_gpu_checkpoint on

In [ ]:
# Feature engineering — FlowBook checkpoints this cell on GPU
df['price_log'] = cudf.Series(np.log1p(df['price'].to_numpy()))
df['weight_sq'] = df['weight'] ** 2
df['price_per_weight'] = df['price'] / df['weight'].clip(lower=1)
print(f"Added 3 features: {df.shape[1]} columns")

In [ ]:
# More feature engineering
for i in range(5):
    df[f'cat_{i}_sq'] = df[f'cat_{i}'] ** 2
    df[f'cat_{i}_x_wt'] = df[f'cat_{i}'] * df['weight']
print(f"Added 10 interaction features: {df.shape[1]} columns")

In [ ]:
# Aggregate features
df['int8_sum'] = sum(df[c].astype(np.int32) for c in int8_cols[:20])
df['f32_mean'] = sum(df[c] for c in f32_cols) / len(f32_cols)
df['f64_std'] = sum((df[c] - df[c].mean()) ** 2 for c in f64_cols[:5]) / 5
print(f"Added 3 aggregate features: {df.shape[1]} columns")

In [ ]:
# Train/test split
mask = rng.random(n_rows) < 0.8
train = df[mask]
test = df[~mask]
print(f"Train: {len(train):,} rows, Test: {len(test):,} rows")
print(f"Total GPU memory: ~{(df.memory_usage(deep=True).sum() + train.memory_usage(deep=True).sum() + test.memory_usage(deep=True).sum()) / 1024**2:.0f} MB")

In [ ]:
# Check FlowBook status — look at State times with GPU checkpointing
%flowbook_status

In [ ]:
%cudf_gpu_checkpoint off
print("GPU checkpoint mode disabled")